# Phase 3: Sportsbook Calibration & Statistical Evaluation
### ECE 143 — NBA Probability Forecasting Project
**Authors:** Zitian & Yu-Jung

---

## Objective
Evaluate the accuracy of traditional sportsbook predictions by:
1. Computing **Brier Score**, **Log Loss**, and **Calibration Analysis**
2. Generating presentation-ready visualizations

**Dataset:** Kaggle NBA sportsbook odds (2008–2025), ~19,820 games with valid moneyline odds.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

%matplotlib inline

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'phase3_evaluation' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.processing.quant_logic import apply_no_vig_probabilities
from phase3_evaluation.metrics import (
    brier_score, log_loss, calibration_bins,
    bootstrap_brier_ci, brier_by_bucket,
)
from phase3_evaluation.plots import (
    plot_calibration_curve, plot_brier_by_bucket,
    plot_segmented_calibration, plot_confidence_tiers,
    plot_brier_by_season,
)

---
## 1. Load & Prepare Data

In [ ]:
# Load raw Kaggle data
raw = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'nba_2008-2025.csv')
print(f'Total games in dataset: {len(raw):,}')
print(f'Seasons: {raw["season"].min()} - {raw["season"].max()}')
print(f'Games with valid moneylines: {raw["moneyline_home"].notna().sum():,}')

# Filter and rename
df = raw.dropna(subset=['moneyline_home', 'moneyline_away']).copy()
df = df.rename(columns={
    'home': 'home_team', 'away': 'away_team',
    'moneyline_home': 'home_odds', 'moneyline_away': 'away_odds',
})

# Apply no-vig conversion (Phase 1 logic)
df = apply_no_vig_probabilities(df)

# Ground truth
df['home_won'] = (df['score_home'] > df['score_away']).astype(int)

# Per-game metrics
eps = 1e-15
df['brier_sportsbook'] = (df['fair_prob_home'] - df['home_won']) ** 2
p_clipped = df['fair_prob_home'].clip(lower=eps, upper=1.0 - eps)
df['logloss_sportsbook'] = -(
    df['home_won'] * np.log(p_clipped) + (1 - df['home_won']) * np.log(1 - p_clipped)
)

print(f'\nReady for evaluation: {len(df):,} games')

---
## 2. Core Metrics

### 2.1 Brier Score
Measures the mean squared error between predicted probabilities and actual outcomes.  
**Formula:** BS = (1/N) * sum((p_i - o_i)^2)  
**Range:** [0, 1]. Lower is better. Random baseline = 0.25.

In [ ]:
prob = df['fair_prob_home'].values
outcome = df['home_won'].values

bs = brier_score(prob, outcome)
ci_lo, ci_hi = bootstrap_brier_ci(prob, outcome)

print(f'Overall Brier Score: {bs:.4f}')
print(f'95% Bootstrap CI:    [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'Random baseline:     0.2500')
print(f'Improvement over random: {((0.25 - bs) / 0.25) * 100:.1f}%')

### 2.2 Log Loss
Penalizes confident but incorrect predictions more heavily than Brier Score.  
**Formula:** LL = -(1/N) * sum( y_i * log(p_i) + (1-y_i) * log(1-p_i) )

In [ ]:
ll = log_loss(prob, outcome)
print(f'Overall Log Loss: {ll:.4f}')
print(f'Random baseline (log(2)): {np.log(2):.4f}')
print(f'Improvement over random: {((np.log(2) - ll) / np.log(2)) * 100:.1f}%')

### 2.3 Calibration Analysis
Groups predictions into probability buckets and compares predicted vs actual win rates.  
Perfect calibration: predicted probability == actual win rate in each bucket.

In [ ]:
cal = calibration_bins(prob, outcome, n_bins=10)

# Add interpretation column
cal['diff'] = cal['mean_actual'] - cal['mean_predicted']
cal['interpretation'] = cal['diff'].apply(
    lambda x: 'overconfident' if x < -0.02 else ('underconfident' if x > 0.02 else 'well-calibrated')
    if not np.isnan(x) else 'no data'
)

print('Calibration Analysis (10 bins):')
print('-' * 80)
for _, row in cal.iterrows():
    if row['count'] > 0:
        print(f"  Bucket {row['bin_center']:.1f}: "
              f"predicted={row['mean_predicted']:.3f}  "
              f"actual={row['mean_actual']:.3f}  "
              f"diff={row['diff']:+.3f}  "
              f"({row['interpretation']})  "
              f"N={int(row['count'])}")

### 2.4 Summary by Segment

In [ ]:
# By Game Type
print('=== By Game Type ===')
for label, mask in [('Regular Season', df['regular'] == True),
                     ('Playoffs', df['playoffs'] == True)]:
    seg = df[mask]
    b = brier_score(seg['fair_prob_home'].values, seg['home_won'].values)
    l = log_loss(seg['fair_prob_home'].values, seg['home_won'].values)
    print(f'  {label:<18}: Brier={b:.4f}  LogLoss={l:.4f}  N={len(seg):,}')

print()

# By Confidence Tier
print('=== By Prediction Confidence ===')
df_c = df.copy()
df_c['max_prob'] = df_c[['fair_prob_home', 'fair_prob_away']].max(axis=1)
for label, mask in [
    ('Strong fav (>=65%)',    df_c['max_prob'] >= 0.65),
    ('Moderate fav (55-65%)', (df_c['max_prob'] >= 0.55) & (df_c['max_prob'] < 0.65)),
    ('Near coin-flip (<55%)', df_c['max_prob'] < 0.55),
]:
    seg = df_c[mask]
    b = brier_score(seg['fair_prob_home'].values, seg['home_won'].values)
    l = log_loss(seg['fair_prob_home'].values, seg['home_won'].values)
    print(f'  {label:<26}: Brier={b:.4f}  LogLoss={l:.4f}  N={len(seg):,}')

---
## 3. Visualizations

### 3.1 Calibration Curve (Reliability Diagram)
The most important chart: X-axis = predicted probability, Y-axis = actual win rate.  
Points on the diagonal = perfect calibration.

In [ ]:
path = plot_calibration_curve(prob, outcome)
display(Image(filename=str(path), width=700))

### 3.2 Brier Score by Probability Bucket
Shows where the sportsbook is most/least accurate.  
Lower bars = better predictions in that confidence range.

In [ ]:
path = plot_brier_by_bucket(prob, outcome)
display(Image(filename=str(path), width=700))

### 3.3 Calibration: Regular Season vs Playoffs

In [ ]:
path = plot_segmented_calibration(df)
display(Image(filename=str(path), width=900))

### 3.4 Calibration by Confidence Tier

In [ ]:
path = plot_confidence_tiers(df)
display(Image(filename=str(path), width=750))

### 3.5 Brier Score Trend by Season (2008-2022)

In [ ]:
path = plot_brier_by_season(df)
display(Image(filename=str(path), width=900))

---
## 4. Key Findings

| Segment | Brier Score | N | Interpretation |
|---|---|---|---|
| **Overall** | **0.2024** [0.2000, 0.2047] | 19,820 | Better than random (0.25) |
| Regular Season | 0.2021 | 18,550 | Consistent calibration |
| Playoffs | 0.2062 | 1,257 | Slightly harder to predict |
| Strong fav (>=65%) | **0.1717** | 11,291 | Best accuracy |
| Moderate fav (55-65%) | 0.2404 | 5,970 | Modest edge |
| Near coin-flip (<55%) | **0.2491** | 2,559 | Near random |

### Summary

1. **Sportsbooks add real value.** Brier Score 0.2024 meaningfully beats the 0.25 random baseline (~19% improvement).

2. **Predictive edge concentrates in clear mismatches.** Strong favorites: Brier=0.1717 (well below random). Coin-flips: Brier=0.2491 (essentially random).

3. **Calibration is stable over time.** No systematic drift across 2008-2022 seasons.

4. **Polymarket API Limitation (Engineering Finding).** The original Polymarket comparison was not feasible due to data purging on closed markets (only 17/495 games matched).